## Adding Human in the Loop Functionality

In [13]:
# Start by loading environment variables (OpenAI model endpoint and key)
import os

from dotenv import load_dotenv
load_dotenv('.env')
OPENAI_API_ENDPOINT=os.getenv("OPENAI_API_ENDPOINT")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT_NAME=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
OPENAI_API_VERSION=os.getenv("OPENAI_API_VERSION")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")

### Adding human-in-the-loop controls to langgraph workflows
Human-in-the-loop allows for execution to pause and resume based on user feedback. The primary interface to this functionality is the interrupt function. Calling interrupt inside a node will pause execution. Execution can be resumed, together with new input from a human, by passing in a Command.

In [14]:
# Import libraries:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

# Command and inturrupt are used for human-in-the-loop:
from langgraph.types import Command, interrupt

In [15]:
# Define the LLM to use in the graph:
llm = init_chat_model(
    model="azure_openai:gpt-5-mini",
    model_provider='azure_openai',
    azure_endpoint=OPENAI_API_ENDPOINT,
    api_key=OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_version=OPENAI_API_VERSION
)

# Define the state structure:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [16]:
# Let's add the human_assistance function to our list of tools:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

tool = TavilySearch(max_results=2)
tools = [tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

In [17]:
# Build the chatbot:
def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    # Because we will be interrupting during tool execution we disable parallel tool calling to avoid
    # repeating any tool invocations when we resume
    assert len(message.tool_calls) <= 1
    return {"messages": [message]}

graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [18]:
# Compile the graph:
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [20]:
# Visualize the updated graph as before:
print(graph.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +---------+         
          | chatbot |         
          +---------+         
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [23]:
# Prompt the chatbot with a question that will engage the new human_assistance tool:
config = {"configurable": {"thread_id": "1"}}

user_input = "I need some expert guidance for building an AI agent. Could you request assistance for me?"
events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some expert guidance for building an AI agent. Could you request assistance for me?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_XUlCW62YMFe5chQGioGbyozA)
 Call ID: call_XUlCW62YMFe5chQGioGbyozA
  Args:
    query: A user has asked for expert guidance to build an AI agent but has not provided implementation details. Please engage directly with the user to collect essential scoping information and then provide a comprehensive, practical plan and hands-on guidance. Start by asking the user the following clarifying questions: 

1) Purpose/goal of the agent (examples: customer support, autonomous robot, research assistant, code-writing assistant).
2) Where it will run / how users will interact with it (web, mobile app, API, robot/embedded device, CLI, other).
3) Key capabilities required (examples: natural language understanding an

In [24]:
# In the previous cell, the agent invoked the human_assistance tool, but then execution is interrupted. To resume,
# we need tp pass a Command object containing data the tool is expecting.
human_response = (
    "We, the experts are here to help! We'd recommend you check out LangGraph to build your agent."
    " It's much more reliable and extensible than simple autonomous agents."
)

human_command = Command(resume={"data": human_response})
events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_XUlCW62YMFe5chQGioGbyozA)
 Call ID: call_XUlCW62YMFe5chQGioGbyozA
  Args:
    query: A user has asked for expert guidance to build an AI agent but has not provided implementation details. Please engage directly with the user to collect essential scoping information and then provide a comprehensive, practical plan and hands-on guidance. Start by asking the user the following clarifying questions: 

1) Purpose/goal of the agent (examples: customer support, autonomous robot, research assistant, code-writing assistant).
2) Where it will run / how users will interact with it (web, mobile app, API, robot/embedded device, CLI, other).
3) Key capabilities required (examples: natural language understanding and generation, code generation, multimodal perception (vision/audio), planning and long-term memory, tool use or action execution, real-time control).
4) Preferred technology